In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import os

In [3]:
#Save file path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)


In [38]:
# Load them back
path = os.path.join(data_dir,"PORTFOLIO_STATS.npz")
data = np.load(path)
mean_returns = data["mean_returns"]
covariance_matrix = data["covariance_matrix"]

print(mean_returns)
print(covariance_matrix)

mean_returns = np.array([0.003, 0.001, 0.001, 0.172])

# covariance matrix
covariance_matrix = np.array([
    [0.0012, 0.0002, 0.0001, 0.0003],
    [0.0002, 0.0004, 0.00005, 0.0001],
    [0.0001, 0.00005, 0.0003, 0.00005],
    [0.0003, 0.0001, 0.00005, 0.005]
])

[0.00118211 0.00126073 0.00098782 0.00304987]
[[0.00039823 0.00037042 0.00028689 0.00041072]
 [0.00037042 0.00109207 0.00038318 0.00083783]
 [0.00028689 0.00038318 0.00036908 0.00044479]
 [0.00041072 0.00083783 0.00044479 0.00115196]]


In [16]:




def monte_carlo_simulation(weights, covariance_matrix, mean_returns, num_simulations = 100000, num_days = 1000, portfolio_value = 10000):
    meanM = np.full(shape = (num_days,len(covariance_matrix)),fill_value = mean_returns)
    
    final_values = []
    L = np.linalg.cholesky(covariance_matrix)
    for i in range(num_simulations):
        Z = np.random.normal(size=(num_days, len(covariance_matrix)))
        
        daily_returns = meanM + Z @ L.T
        weighted_returns = daily_returns @ weights
        portfolio_final_value = portfolio_value * np.prod(1 + weighted_returns)
        final_values.append(portfolio_final_value)

    final_values = np.array(final_values)
    final_mean = np.mean(final_values)
    final_std = np.std(final_values)
    
    return final_mean, final_std

In [17]:
def negative_sharpe_ratio(weights, covariance_matrix, mean_returns):
    mean, std = monte_carlo_simulation(weights,covariance_matrix, mean_returns)
    return - mean / std


In [39]:
n_assets = len(mean_returns)

constraints = {"type": "eq", "fun": lambda n: np.sum(n)-1}

bounds = [(0,1) for i in range(n_assets)]

initial_guess = [0.85, 0.05, 0.05,0.05]
#np.repeat(1/n_assets, n_assets)


opt_result = minimize(negative_sharpe_ratio, initial_guess, args = (covariance_matrix, mean_returns), bounds = bounds, constraints = constraints, method = 'SLSQP', options={'ftol': 1e-9, 'disp': True})
opt_weights = opt_result.x
print("Optimal portfolio weights:", opt_weights)

KeyboardInterrupt: 

In [20]:
print(mean_returns)
print(covariance_matrix)

[0.00118211 0.00126073 0.00098782 0.00304987]
[[0.00039823 0.00037042 0.00028689 0.00041072]
 [0.00037042 0.00109207 0.00038318 0.00083783]
 [0.00028689 0.00038318 0.00036908 0.00044479]
 [0.00041072 0.00083783 0.00044479 0.00115196]]
